# Exercise 3 Digitization and Data Analytics - Machine Learning

## Supervised Learning - Bayes Classifier

Content

- Fundamental concepts in interaction with Apache Spark
- Usage of basic algorithms (Bayes Classifier) from the machine library MLLib within Spark

In [1]:
!pip install pyspark

In [2]:
%matplotlib inline
import os


# Adapt the path to point to your iris.scale and iris.csv
scaledIrisDataPath = "iris.scale"
irisDataPath = "iris.csv"


import platform
print('the python version is: ' + platform.python_version())
import pyspark
from pyspark import SparkContext

the python version is: 3.12.13


### Load the Bayes Classifier library:

In [ ]:
# $example on$
from pyspark.ml import Pipeline
from pyspark.ml.classification import NaiveBayes, DecisionTreeClassifier
from pyspark.ml.feature import StringIndexer, VectorIndexer, VectorAssembler, IndexToString
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
# $example off$
from pyspark.sql import SparkSession
from pyspark.sql.types import DoubleType, StructType, StructField, StringType


In [4]:
spark = SparkSession.builder.appName("NaiveBayesClassificationExample").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/09 23:16:36 WARN Utils: Your hostname, Krishnas-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.155 instead (on interface en0)
26/06/09 23:16:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/09 23:16:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Load your data and preprocess it as already done in previous examples. 

In [5]:
# generate DataFrame, use DataFrameReader class to parse csv input
dataframe=spark.read.csv(irisDataPath,
                    schema=StructType([StructField("sepal_length",DoubleType(),True),
                                       StructField("sepal_width",DoubleType(),True),
                                       StructField("petal_length",DoubleType(),True),
                                       StructField("petal_width",DoubleType(),True),
                                       StructField("spezies",StringType(),True)]))
#define a corresponding vector assembler
assembler = VectorAssembler(inputCols=["sepal_length",
                                 "sepal_width",
                                 "petal_length",
                                 "petal_width"],
                      outputCol="features")
dataset = assembler.transform(dataframe)
dataset.printSchema()
dataset.show(15, False)

root
 |-- sepal_length: double (nullable = true)
 |-- sepal_width: double (nullable = true)
 |-- petal_length: double (nullable = true)
 |-- petal_width: double (nullable = true)
 |-- spezies: string (nullable = true)
 |-- features: vector (nullable = true)

+------------+-----------+------------+-----------+-----------+-----------------+
|sepal_length|sepal_width|petal_length|petal_width|spezies    |features         |
+------------+-----------+------------+-----------+-----------+-----------------+
|5.1         |3.5        |1.4         |0.2        |Iris-setosa|[5.1,3.5,1.4,0.2]|
|4.9         |3.0        |1.4         |0.2        |Iris-setosa|[4.9,3.0,1.4,0.2]|
|4.7         |3.2        |1.3         |0.2        |Iris-setosa|[4.7,3.2,1.3,0.2]|
|4.6         |3.1        |1.5         |0.2        |Iris-setosa|[4.6,3.1,1.5,0.2]|
|5.0         |3.6        |1.4         |0.2        |Iris-setosa|[5.0,3.6,1.4,0.2]|
|5.4         |3.9        |1.7         |0.4        |Iris-setosa|[5.4,3.9,1.7,0.4]|
|4.

#### Deal with Categorical Labels and Variables

#### Exercise:

Define a transformation, that adds indices to the labels in your data (use StringIndexer), and fit it to your data.
Why it is necessary to have a "fit" and a "transform" step to obtain the labels in the indexed manner?

In [6]:
# Convert target into numerical categories
labelIndexer = StringIndexer(inputCol="spezies", outputCol="indexedLabel")

In [7]:
labelIndexerModel = labelIndexer.fit(dataframe)
dataframeLabelIndexed = labelIndexerModel.transform(dataframe)
dataframeLabelIndexed.printSchema()
dataframeLabelIndexed.show(15, False)

root
 |-- sepal_length: double (nullable = true)
 |-- sepal_width: double (nullable = true)
 |-- petal_length: double (nullable = true)
 |-- petal_width: double (nullable = true)
 |-- spezies: string (nullable = true)
 |-- indexedLabel: double (nullable = false)

+------------+-----------+------------+-----------+-----------+------------+
|sepal_length|sepal_width|petal_length|petal_width|spezies    |indexedLabel|
+------------+-----------+------------+-----------+-----------+------------+
|5.1         |3.5        |1.4         |0.2        |Iris-setosa|0.0         |
|4.9         |3.0        |1.4         |0.2        |Iris-setosa|0.0         |
|4.7         |3.2        |1.3         |0.2        |Iris-setosa|0.0         |
|4.6         |3.1        |1.5         |0.2        |Iris-setosa|0.0         |
|5.0         |3.6        |1.4         |0.2        |Iris-setosa|0.0         |
|5.4         |3.9        |1.7         |0.4        |Iris-setosa|0.0         |
|4.6         |3.4        |1.4         |0.3 

#### Why fit() and transform()?
`StringIndexer` is an estimator that must first learn the mapping from string labels to numeric indices with `fit()`. After the mapping is learned, `transform()` applies the same encoding consistently to training and test data.

#### Training and Test Data Sets

#### Exercise:
Split your data into training and test data.

In [8]:
# Split dataset randomly into Training and Test sets (30% held out for testing). Set seed for reproducibility
(trainingData, testData) = dataframe.randomSplit([0.7, 0.3], seed = 100)

print(trainingData.count())
print(testData.count())

104
46


#### Exercise:
Use the above defined label indexer to show the indexed test- and training data.

In [ ]:
trainDataLabelIndexed = labelIndexerModel.transform(trainingData)
testDataLabelIndexed = labelIndexerModel.transform(testData)
trainDataLabelIndexed.select("spezies", "indexedLabel", "sepal_length", "sepal_width", "petal_length", "petal_width").show(10, False)
testDataLabelIndexed.select("spezies", "indexedLabel", "sepal_length", "sepal_width", "petal_length", "petal_width").show(10, False)

#### Train a Naive Bayes Classifier

#### Exercise:
Define a Naive Bayes classifier for the iris data set. Have a look at the documentation of  NaiveBayes() and its parameters:  
1. https://spark.apache.org/docs/2.2.0/api/python/_modules/pyspark/ml/classification.html  
2. https://spark.apache.org/docs/2.2.0/api/python/pyspark.ml.html?highlight=naivebayes#pyspark.ml.classification.NaiveBayes).

Hint: Pay attention to the labels within your data, you use for the classifier.

In [ ]:
nb = NaiveBayes(featuresCol="features", labelCol="indexedLabel", predictionCol="prediction", smoothing=1.0, modelType="multinomial")

#### Exercise:
1. Read the documentation about the pipeline-component (see https://spark.apache.org/docs/latest/ml-pipeline.html#pipeline-components).
2. Define a pipeline, describing the process of indexing and transforming your data, and then training the naive bayes classifier.
3. Train the model.

In [ ]:
pipeline = Pipeline(stages=[assembler, labelIndexer, nb])

In [ ]:
# Run stages in pipeline and train model
model = pipeline.fit(dataframe)

#### Use the trained model for predictions

In [ ]:
# Make predictions.
predictions = model.transform(testData)

In [ ]:
# Select example rows to display.
predictions.select("prediction", "indexedLabel", "spezies", "features").show(15, False)

AttributeError: 'ellipsis' object has no attribute 'select'

In [ ]:
# Show the same with the original labels
converter = IndexToString(inputCol="prediction", outputCol="predictedLabel", labels=labelIndexerModel.labels)
convertedPrediction = converter.transform(predictions)
convertedPrediction.select("prediction", "predictedLabel", "indexedLabel", "spezies", "features").show(15, False)

#### Evaluate your results

In [ ]:
# Select (prediction, true label) and compute test error
accuracy_evaluator = MulticlassClassificationEvaluator(labelCol="indexedLabel", predictionCol="prediction", metricName="accuracy")
f1_evaluator = MulticlassClassificationEvaluator(labelCol="indexedLabel", predictionCol="prediction", metricName="f1")
precision_evaluator = MulticlassClassificationEvaluator(labelCol="indexedLabel", predictionCol="prediction", metricName="weightedPrecision")
recall_evaluator = MulticlassClassificationEvaluator(labelCol="indexedLabel", predictionCol="prediction", metricName="weightedRecall")

accuracy = accuracy_evaluator.evaluate(predictions)
precision = precision_evaluator.evaluate(predictions)
recall = recall_evaluator.evaluate(predictions)
f1_score = f1_evaluator.evaluate(predictions)

print("Accuracy = %g" % accuracy)
print("Test Error = %g " % (1.0 - accuracy))
print("Weighted Precision = %g" % precision)
print("Weighted Recall = %g" % recall)
print("Weighted F1 = %g" % f1_score)

#### Results and answers

- Using a 70/30 split gives about 105 training rows and 45 test rows for the Iris dataset. That is enough to keep the error in a reasonable interval for this simple, well-separated dataset.
- When you change the metric, you can observe:
  - `accuracy`: overall fraction of correct predictions,
  - `weightedPrecision`: precision averaged across classes,
  - `weightedRecall`: recall averaged across classes,
  - `f1`: harmonic mean of precision and recall.
- For Iris, accuracy, weighted precision, weighted recall, and F1 are often very similar for a good model, which shows that the classifier is balanced across classes.

In [ ]:
spark.stop()